In [1]:
import os
import sys
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')
# setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# defining the class mappings, could have been done simply by using image folders but forgot to do that
Class_Mapping = {
    'AnnualCrop': 0, 'Forest': 1, 'HerbaceousVegetation': 2,
    'Highway': 3, 'Industrial': 4, 'Pasture': 5,
    'PermanentCrop': 6, 'Residential': 7, 'River': 8, 'SeaLake': 9
}

#downloading training and testing data
train_zip_url = "https://github.com/ishananand06/CSOT26_Computer-Vision/raw/3aa9e667e766ed18e78f716dd0d081f9b2572873/Week%202/train.zip"
if os.path.exists('train.zip'):
    os.remove('train.zip')
os.makedirs('EuroSAT/train_files', exist_ok=True)
!wget -q {train_zip_url} -O train.zip
!unzip -q -o train.zip -d EuroSAT/train_files
print("Training data downloaded and unzipped.")


test_zip_url = "https://github.com/ishananand06/CSOT26_Computer-Vision/raw/3aa9e667e766ed18e78f716dd0d081f9b2572873/Week%202/test_set.zip"
if os.path.exists('test_set.zip'):
    os.remove('test_set.zip')
os.makedirs('EuroSAT/test_files', exist_ok=True)
!wget -q {test_zip_url} -O test_set.zip
!unzip -q -o test_set.zip -d EuroSAT/test_files
print("Test data downloaded and unzipped.")

Mounted at /content/drive
Using device: cuda
Training data downloaded and unzipped.
Test data downloaded and unzipped.


In [2]:
#class for preprocessing of labelled data from train.zip
class EuroSATLabeled(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.data = []
        for class_name in os.listdir(self.img_dir):
            class_path = os.path.join(self.img_dir, class_name)
            if os.path.isdir(class_path):
                class_label = Class_Mapping[class_name]
                for img_name in os.listdir(class_path):
                      self.data.append((os.path.join(class_path, img_name), class_label, img_name))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, class_label, img_filename = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, class_label, img_filename

class EuroSATUnlabeled(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.data = []
        for img_name in os.listdir(self.img_dir):
              self.data.append((os.path.join(self.img_dir, img_name), img_name))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, img_filename = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, img_filename

train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=(0, 360)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
full_train_aug = EuroSATLabeled(img_dir='EuroSAT/train_files/train', transforms=train_transforms)
full_train_no_aug = EuroSATLabeled(img_dir='EuroSAT/train_files/train', transforms=test_transforms)

targets = [item[1] for item in full_train_aug.data]
indices = list(range(len(full_train_aug)))

train_indices, val_indices = train_test_split(
    indices, test_size=0.2, stratify=targets, random_state=42
)

train_dataset = Subset(full_train_aug, train_indices)
val_dataset = Subset(full_train_no_aug, val_indices)
test_dataset = EuroSATUnlabeled(img_dir='EuroSAT/test_files/test_set', transforms=test_transforms)

batch_size = 256

# --- [UPDATED] Hardware Acceleration for DataLoaders ---
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True,persistent_workers=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Submission samples: {len(test_dataset)}")

Training samples: 18400
Validation samples: 4600
Submission samples: 4000


In [4]:
import torch.nn.functional as F

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != self.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.linear(out)
        return out

def build_resnet34_64x64(num_classes):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes=num_classes)

model = build_resnet34_64x64(num_classes=len(Class_Mapping)).to(device)

def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)


model.apply(init_weights)
print("Applied Kaiming Normal initialization.")

num_epochs=70
learning_rate = 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4, amsgrad=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
model = torch.compile(model)
print("Model successfully compiled and optimized.")

Applied Kaiming Normal initialization.
Model successfully compiled and optimized.


In [5]:
scaler = torch.amp.GradScaler('cuda')
cutmix = v2.CutMix(num_classes=len(Class_Mapping), alpha=1.0)

def train_loop(dataloader, model, loss_fn, optimizer, cutmix=None):
    model.train()
    size = len(dataloader.dataset)
    correct = 0

    for batch, (X, y, _) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        if cutmix is not None:
            X, y = cutmix(X, y)

        optimizer.zero_grad()

        # --- Run the forward pass in 16-bit Mixed Precision ---
        with torch.amp.autocast('cuda'):
            pred = model(X)
            loss = loss_fn(pred, y)

        # --- Scale the loss and update gradients ---
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if batch % 10 == 0: # Prints more often for larger batch sizes
            loss_val = loss.item()
            current = batch * dataloader.batch_size + len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")

        if cutmix is not None:
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
        else:
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    train_accuracy = correct / size
    print(f"Training Accuracy: {(100*train_accuracy):>0.1f}%")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y, _ in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss

In [6]:

checkpoint_path = '/content/drive/MyDrive/EuroSAT_Best_ResNet34.pth'
best_vloss = float('inf')
start_epoch = 0


for t in range(start_epoch, num_epochs):
    print(f"Epoch {t+1}")
    train_loop(train_dataloader, model, loss_fn, optimizer, cutmix=cutmix)
    vloss = test_loop(val_dataloader, model, loss_fn)


    scheduler.step()


    if vloss < best_vloss:
        print(f"Validation loss decreased from {best_vloss:.6f} to {vloss:.6f}. Saving checkpoint...")
        best_vloss = vloss
        torch.save({
            'epoch': t + 1,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_vloss': best_vloss
        }, checkpoint_path)

Epoch 1


W0612 13:12:28.947000 1998 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


loss: 2.432981  [  256/18400]
loss: 1.889452  [ 2816/18400]
loss: 1.280112  [ 5376/18400]
loss: 1.624180  [ 7936/18400]
loss: 1.710468  [10496/18400]
loss: 1.696301  [13056/18400]
loss: 1.518688  [15616/18400]
loss: 1.657898  [18176/18400]
Training Accuracy: 50.0%
Validation Error: 
 Accuracy: 59.9%, Avg loss: 1.229780 

Validation loss decreased from inf to 1.229780. Saving checkpoint...
Epoch 2
loss: 1.127247  [  256/18400]
loss: 1.232584  [ 2816/18400]
loss: 1.617331  [ 5376/18400]
loss: 1.531160  [ 7936/18400]
loss: 1.601688  [10496/18400]
loss: 1.479847  [13056/18400]
loss: 1.473098  [15616/18400]
loss: 0.907357  [18176/18400]
Training Accuracy: 62.5%
Validation Error: 
 Accuracy: 66.8%, Avg loss: 1.003809 

Validation loss decreased from 1.229780 to 1.003809. Saving checkpoint...
Epoch 3
loss: 1.479418  [  256/18400]
loss: 1.416063  [ 2816/18400]
loss: 1.341105  [ 5376/18400]
loss: 1.396455  [ 7936/18400]
loss: 1.337922  [10496/18400]
loss: 1.346400  [13056/18400]
loss: 1.095056 

In [7]:

print("\nLoading the best checkpoint for final evaluation...")
model.load_state_dict(torch.load(checkpoint_path)['model_state'])

print("\n--- Generating Final Validation Report ---")
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for X, y, _ in val_dataloader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        preds = outputs.argmax(dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(Class_Mapping.keys())))

print("\n--- Generating Predictions for Submission ---")
submission_data = []

with torch.no_grad():
    for images, img_names in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        for name, p_idx in zip(img_names, preds):
            submission_data.append({
                'image_id': name,
                'label': p_idx
            })

submission_df = pd.DataFrame(submission_data)
submission_df = submission_df.sort_values(by='image_id').reset_index(drop=True)

csv_filename = '2025MT11352_CNN.csv'
submission_df.to_csv(csv_filename, index=False)

print(f"Submission safely generated and saved to {csv_filename}")



Loading the best checkpoint for final evaluation...

--- Generating Final Validation Report ---

Confusion Matrix:
[[515   2   1   0   0   0   1   0   1   0]
 [  0 518   2   0   0   0   0   0   0   0]
 [  1   1 512   0   0   2   4   0   0   0]
 [  0   0   0 416   1   0   0   1   2   0]
 [  0   0   0   2 418   0   0   0   0   0]
 [  3   0   4   0   0 312   0   0   0   1]
 [  5   0   9   2   2   2 399   0   1   0]
 [  0   0   0   2   4   0   0 514   0   0]
 [  3   0   0   2   1   1   0   1 412   0]
 [  1   0   0   0   1   0   0   0   0 518]]

Classification Report:
                      precision    recall  f1-score   support

          AnnualCrop       0.98      0.99      0.98       520
              Forest       0.99      1.00      1.00       520
HerbaceousVegetation       0.97      0.98      0.98       520
             Highway       0.98      0.99      0.99       420
          Industrial       0.98      1.00      0.99       420
             Pasture       0.98      0.97      0.98     

In [8]:
save_path = 'EuroSAT_resnet34.pth'
torch.save(model.state_dict(), save_path)